In [1]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_excel('MSC_ADITI_CLEANED_DATA.xlsx')
data.head()

,adjusted_date,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,distance,actual_total_fo,total_co_2,activity_time,status
0,2017-07-09,7.27,6.7,3.0,1.1,2.2,2,0,8.00,1.3,4.05,1.1,SAIL
1,2017-07-09,18.00,6.7,3.0,11.0,11.0,1,0,198.00,26.5,82.52,11.0,SAIL
2,2017-07-10,17.27,6.7,3.0,11.0,0.0,0,0,189.97,25.8,80.34,11.0,SAIL
3,2017-07-10,7.50,6.7,0.0,3.2,6.4,2,0,24.00,4.0,12.46,3.2,SAIL
4,2017-07-11,9.41,7.6,3.0,1.7,3.4,2,0,16.00,3.2,9.96,1.7,SAIL


In [4]:
port_data = data[data['status'] == 'PORT']
sail_data = data[data['status'] == 'SAIL']

port_data.shape, sail_data.shape

((1472, 13), (4858, 13))

In [7]:
def run_xgboost(X, y):
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.metrics import mean_squared_error
    from xgboost import XGBRegressor
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_squared_error, r2_score


    # Split the data into training and testing sets for Deep Learning
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)



    # Initialize and train the XGBoost model
    xgb = XGBRegressor()
    xgb.fit(X_train, y_train)

    # Predictions
    y_pred = xgb.predict(X_test)

    # Evaluation
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'Mean Squared Error: {mse}')
    print(f'R-squared: {r2}') 
    return xgb

# Sailing CO2 Prediction Model

In [10]:
sail_data.columns

Index(['adjusted_date', 'speed', 'mean_draft', 'sea_state',
       'me_actual_steaming_time', 'ae_t_steaming', 'aux_running',
       'blr_running', 'distance', 'actual_total_fo', 'total_co_2',
       'activity_time', 'status'],
      dtype='object')

In [12]:
sail_data['new_ss'] = 10 - sail_data['sea_state']


C:\Users\akshaya\AppData\Local\Temp\ipykernel_152652\1320236166.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sail_data['new_ss'] = 10 - sail_data['sea_state']


In [14]:
# Separate features and predictor
X = sail_data[[  'speed', 'mean_draft', 'new_ss',
       'me_actual_steaming_time', 
               'ae_t_steaming', 'aux_running',
       'blr_running'  
              ]
          ]
y = sail_data['total_co_2']     

In [20]:
sail_co2_model = run_xgboost(X, y)
sail_co2_model

Mean Squared Error: 230.27618965673415
R-squared: 0.9438681066158532


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

# Sailing Distance Prediction Model

In [23]:
# Separate features and predictor
X = sail_data[[  'speed', 'mean_draft', 'new_ss',
       'me_actual_steaming_time', 
               'ae_t_steaming', 'aux_running',
       'blr_running' ]
          ]
y = sail_data['distance']  

In [25]:
sail_distance_model = run_xgboost(X, y)
sail_distance_model

Mean Squared Error: 27.230933850543558
R-squared: 0.998620985005653


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

# Port CO2 Prediction Model

In [28]:
port_data.columns

Index(['adjusted_date', 'speed', 'mean_draft', 'sea_state',
       'me_actual_steaming_time', 'ae_t_steaming', 'aux_running',
       'blr_running', 'distance', 'actual_total_fo', 'total_co_2',
       'activity_time', 'status'],
      dtype='object')

In [30]:
# Separate features and predictor
X = port_data[[ 
    'mean_draft',  
    'ae_t_steaming', 
    'aux_running',
       'blr_running',  
    'activity_time'
]
          ]
y = port_data['total_co_2']  

In [42]:
port_co2_model = run_xgboost(X, y)
port_co2_model

Mean Squared Error: 6.784588897568613
R-squared: 0.8677791832677808


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

# Subset Port Data CO2  Prediction

In [45]:
subset_port_df = pd.read_excel("Subset_PORT_DF.xlsx")
subset_port_df.head()

,mean_draft,ae_t_steaming,aux_running,blr_running,activity_time,Cluster
0,5.45,0.0,0,0,24.0,1
1,5.35,7.9,2,0,5.4,1
2,6.60,1.9,1,1,1.9,1
3,7.45,11.9,2,0,8.9,1
4,5.45,0.0,0,0,24.0,1


In [47]:
subset_port_df.columns

Index(['mean_draft', 'ae_t_steaming', 'aux_running', 'blr_running',
       'activity_time', 'Cluster'],
      dtype='object')

In [49]:
predicted_co2_port = {}
for i in subset_port_df['Cluster'].unique():
    df = subset_port_df[subset_port_df['Cluster'] == i][['mean_draft', 'ae_t_steaming', 'aux_running', 'blr_running',
       'activity_time' ]]
 
    port_co2_predicted = port_co2_model.predict(df).sum()
    predicted_co2_port['Port_Cluster_' + str(i)] = round(port_co2_predicted, 2)
predicted_co2_port

{'Port_Cluster_1': np.float32(250.85),
 'Port_Cluster_2': np.float32(815.17),
 'Port_Cluster_3': np.float32(617.13)}

# Subset Sailing  Data CO2 and Distance Prediction

In [52]:
subset_sail_df = pd.read_excel("Subset_SAIL_DF.xlsx")
subset_sail_df.head()

,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,Cluster
0,12.00,8.35,3,0.5,4.0,2,0,1
1,8.83,9.55,2,6.0,12.0,2,1,1
2,8.46,10.15,3,1.3,6.0,3,1,1
3,14.76,6.35,2,2.1,9.5,3,1,1
4,5.71,10.65,0,1.4,7.6,3,0,1


In [54]:
subset_sail_df['new_ss'] = 10 - subset_sail_df['sea_state']

In [56]:
subset_sail_df.columns

Index(['speed', 'mean_draft', 'sea_state', 'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running', 'Cluster', 'new_ss'],
      dtype='object')

In [58]:
predicted_co2_sail = {}
for i in subset_sail_df['Cluster'].unique():
    df = subset_sail_df[subset_sail_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss',
                                                         'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_co2_predicted = sail_co2_model.predict(df).sum()
    predicted_co2_sail['Sail_Cluster_' + str(i)] = round(sail_co2_predicted, 2)
predicted_co2_sail

{'Sail_Cluster_1': np.float32(7306.9),
 'Sail_Cluster_2': np.float32(7681.34),
 'Sail_Cluster_3': np.float32(11330.1)}

In [60]:
predicted_distance_sail = {}
for i in subset_sail_df['Cluster'].unique():
    df = subset_sail_df[subset_sail_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss',
                                                         'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_distance_predicted = sail_distance_model.predict(df).sum()
    predicted_distance_sail['Sail_Cluster_' + str(i)] = round(sail_distance_predicted, 2)
predicted_distance_sail

{'Sail_Cluster_1': np.float32(16670.19),
 'Sail_Cluster_2': np.float32(15666.07),
 'Sail_Cluster_3': np.float32(27604.88)}

In [62]:
deadweight = 39228
required_cii_2024  = 10.46 

In [64]:
# Subset 1 CII

attained_cii_1 = ((predicted_co2_sail['Sail_Cluster_1'] + predicted_co2_port['Port_Cluster_1'] )   * 1000000)/(deadweight * predicted_distance_sail['Sail_Cluster_1'])
attained_cii_1

np.float32(11.557284)

In [66]:
a_r_1 = round(attained_cii_1/required_cii_2024, 2)
a_r_1

np.float32(1.1)

# D - Rating

In [69]:
# Subset 1 CII

attained_cii_2 = ((predicted_co2_sail['Sail_Cluster_2'] + predicted_co2_port['Port_Cluster_2'] )   * 1000000)/(deadweight * predicted_distance_sail['Sail_Cluster_2'])
attained_cii_2

np.float32(13.825611)

In [71]:
a_r_2 = round(attained_cii_2/required_cii_2024, 2)
a_r_2

np.float32(1.32)

# C - Rating

In [74]:
# Subset 1 CII

attained_cii_3 = ((predicted_co2_sail['Sail_Cluster_3'] + predicted_co2_port['Port_Cluster_3'] )   * 1000000)/(deadweight * predicted_distance_sail['Sail_Cluster_3'])
attained_cii_3

np.float32(11.032785)

In [76]:
a_r_3 = round(attained_cii_3/required_cii_2024, 2)
a_r_3

#CII_RATIO = ATT_CII / CII_REQ

np.float32(1.05)

# D - Rating

In [80]:
d1 =  0.83
d2 =  0.94
d3 =  1.07
d4 =  1.19


if ratio  <= d1:
    return "A"         
elif (ratio > d1) & (ratio <= d2):
    return "B"
elif (ratio > d2) & (ratio <= d3):
    return "C"
elif (ratio > d3) & (ratio <= d4):
    return "D"
elif (ratio > d4) :
    return "E" 

SyntaxError: 'return' outside function (2837804368.py, line 8)

# 5% Reduction in Speed

In [83]:
sail_5pct_df = subset_sail_df.copy()
sail_5pct_df['speed'] = sail_5pct_df['speed'].apply(lambda x : x*0.95)
sail_5pct_df

,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,Cluster,new_ss
0,11.4000,8.35,3,0.5,4.0,2,0,1,7
1,8.3885,9.55,2,6.0,12.0,2,1,1,8
2,8.0370,10.15,3,1.3,6.0,3,1,1,7
3,14.0220,6.35,2,2.1,9.5,3,1,1,8
4,5.4245,10.65,0,1.4,7.6,3,0,1,10
...,...,...,...,...,...,...,...,...,...
1112,13.9745,8.60,4,3.4,8.4,3,0,3,6
1113,16.7200,11.75,3,7.5,15.0,2,0,3,7
1114,10.9630,10.20,3,6.5,28.8,2,0,3,7
1115,14.0030,5.35,2,1.9,1.9,1,0,3,8


In [85]:
predicted_co2_sail_5pct = {}

for i in sail_5pct_df['Cluster'].unique():
    df = sail_5pct_df[sail_5pct_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss', 
                                                     'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_co2_predicted = sail_co2_model.predict(df).sum()
    predicted_co2_sail_5pct['Sail_Cluster_' + str(i)] = round(sail_co2_predicted, 2)
predicted_co2_sail_5pct

{'Sail_Cluster_1': np.float32(7023.74),
 'Sail_Cluster_2': np.float32(7331.93),
 'Sail_Cluster_3': np.float32(10336.91)}

In [87]:
predicted_distance_sail_5pct = {}

for i in sail_1pct_df['Cluster'].unique():
    df = sail_1pct_df[sail_1pct_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss',
                                                     'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_distance_predicted = sail_distance_model.predict(df).sum()
    predicted_distance_sail_5pct['Sail_Cluster_' + str(i)] = round(sail_distance_predicted, 2)
predicted_distance_sail_5pct

NameError: name 'sail_1pct_df' is not defined

# 10% Reduction in Speed

In [ ]:
sail_10pct_df = subset_sail_df.copy()
sail_10pct_df['speed'] = sail_10pct_df['speed'].apply(lambda x : x*0.90)
sail_10pct_df

In [ ]:
predicted_co2_sail_10pct = {}

for i in sail_10pct_df['Cluster'].unique():
    df = sail_10pct_df[sail_10pct_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss', 
                                                     'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_co2_predicted = sail_co2_model.predict(df).sum()
    predicted_co2_sail_10pct['Sail_Cluster_' + str(i)] = round(sail_co2_predicted, 2)
predicted_co2_sail_10pct

In [ ]:
predicted_distance_sail_10pct = {}

for i in sail_10pct_df['Cluster'].unique():
    df = sail_10pct_df[sail_10pct_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss',
                                                     'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_distance_predicted = sail_distance_model.predict(df).sum()
    predicted_distance_sail_10pct['Sail_Cluster_' + str(i)] = round(sail_distance_predicted, 2)
predicted_distance_sail_10pct

# 15% Reduction in Speed

In [ ]:
sail_15pct_df = subset_sail_df.copy()
sail_15pct_df['speed'] = sail_15pct_df['speed'].apply(lambda x : x*0.85)
sail_15pct_df

In [ ]:
predicted_co2_sail_15pct = {}

for i in sail_15pct_df['Cluster'].unique():
    df = sail_15pct_df[sail_15pct_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss', 
                                                     'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_co2_predicted = sail_co2_model.predict(df).sum()
    predicted_co2_sail_15pct['Sail_Cluster_' + str(i)] = round(sail_co2_predicted, 2)
predicted_co2_sail_15pct

In [ ]:
predicted_distance_sail_15pct = {}

for i in sail_15pct_df['Cluster'].unique():
    df = sail_15pct_df[sail_15pct_df['Cluster'] == i][['speed', 'mean_draft', 'new_ss',
                                                     'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running' ]]
 
    sail_distance_predicted = sail_distance_model.predict(df).sum()
    predicted_distance_sail_15pct['Sail_Cluster_' + str(i)] = round(sail_distance_predicted, 2)
predicted_distance_sail_15pct

In [ ]:
{'imo': 9311737,
 'deadweight': 39228,
 'vessel_speed_rolling': 10.01,
 'avg_speed_rolling': 11.3,
 'distance_rolling': 201.19,
 'sea_state_rolling': 3.0,
 'mean_draft_rolling': 9.43,
 'me_con_rolling': 23.33,
 'ae_con_rolling': 3.98,
 'bl_con_rolling': 1.37,
 'actual_total_fo_rolling': 28.68,
 'total_co_2_rolling': 90.38,
 'me_actual_steaming_time_rolling': 16.44,
 'ae_t_steaming_rolling': 25.92,
 'year_to_end_days': 189,
 'remaining_days_2024': 177,
 'distance_2024': 26855.6,
 'total_co_2_2024': 13529.08,
 'attained_cii_2024': 12.84,
 'attained_cii_sailing_2024': 11.81,
 'pattern_steaming_days': 229.54,
 'pattern_port_days': 105.46,
 'expected_sailing_days': 121.0,
 'expected_port_days': 56.0,
 'expected_distance': 24343.99,
 'model_predicted_distance_per_day_sailing': 165.10315,
 'expected_total_distance_predicted': 19977.48,
 'total_projected_co_2': 10935.98,
 'predicted_co2_sailing_per_day': 46.80456,
 'predicted_co2_port_per_day': 19.563286,
 'total_predicted_co2_sailing': 5663.351955413818,
 'total_predicted_co2_port': 1095.5440063476562,
 'total_predicted_co2': 6758.9,
 'predicted_calculated_co2_per_day': 38.19,
 'projected_cii': 12.18,
 'predicted_cii': 11.04,
 'predicted_cii_sailing': 7.23,
 'total_data_rows': 335,
 'rolling_days': 335,
 'required_cii_2024': 10.46,
 'a_r_ratio': 1.23,
 'a_r_ratio_sailing': 1.13,
 'a_r_ratio_projected': 1.16,
 'a_r_ratio_predicted': 1.06,
 'a_r_ratio_predicted_sailing': 0.69,
 'rating_2024': 'E',
 'rating_projected': 'D',
 'rating_predicted': 'C'}

In [ ]:
!pip install pybind11>=2.12


In [ ]:
!pip install numpy

In [ ]:
!conda install numpy=1.24  # Or another version that is <2


In [ ]:
import numpy
print(numpy.__version__)


In [ ]:
!conda install numpy=1.24
